# Bài 1: Titanic Dataset
## Sử dụng lại bộ dữ liệu Titanic trong bài tập về nhà trước.

## Yêu cầu
Sử dụng Logistic Regression để dự đoán hành khách sống sót hay không.
Sử dụng cùng cách chia dữ liệu train/test như bài tập trước nếu có thể.
Đánh giá kết quả của mô hình Logistic Regression.
So sánh kết quả của Logistic Regression với kết quả của Linear Regression trong bài tập trước.

## Nội dung so sánh
Có thể so sánh dựa trên các chỉ số:
Accuracy
Precision
Recall
F1-score
Confusion Matrix

## Đưa ra nhận xét về mô hình phù hợp hơn đối với bài toán phân loại hành khách sống sót.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

In [ ]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

## Chuẩn bị dữ liệu.
Lựa chọn biến đầu vào (features) và biến cần dự đoán (target).

In [ ]:
features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]
X = df[features] # ma trận dữ liệu
y = df["survived"] # nhãn

## Chia tập Train/ Test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Chia dữ liệu số và dữ liệu phân loại.
Mỗi kiểu dữ liệu cần xử lý khác nhau.

In [ ]:
# Tiền xử lý dữ liệu
numeric_features = [
    "age",
    "fare",
    "sibsp",
    "parch",
    "pclass"
] # dữ liệu có giá trị số
categorical_features = [
    "sex",
    "embarked"
] # dữ liệu dạng chữ
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # thay giá trị thiếu thành Median
    ("scaler", StandardScaler()) # chuẩn hóa dữ liệu cho cùng thang đo
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")), # thay giá trị thiếu thành giá trị xuất hiện nhiều nhất
    ("onehot", OneHotEncoder(handle_unknown="ignore")), # chuyển thành tệp số 0 và 1
])
preprocessor = ColumnTransformer([ # áp dụng tiền xử lý cho từng nhóm đặc trưng khác nhau, sau đó kết hợp thành dữ liệu hoàn chỉnh
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

Pipeline làm theo trình tự:
- Dữ liệu.
- Điền giá trị thiếu.
- Chuẩn hóa.
- Hoàn thành.

## Xây dựng Logistic Regression.

In [ ]:
# Mô hình Logistic Regression
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])
logistic_model.fit(X_train, y_train) 
# mô hình sẽ học những mối quan hệ với survived để tìm ra các hệ số tối ưu.

Pipeline làm theo trình tự:
- Dữ liệu.
- Tiền xử lý.
- Logistic Regression.
- Dự đoán.
Mô hình Logistic Regression giúp dự đoán khả năng sống sót.

In [ ]:
# Dự đoán
y_pred = logistic_model.predict(X_test)

In [ ]:
# Đánh giá mô hình
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
acc = accuracy_score(y_test, y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1_score:", f1)
print("Classification Report:", classification_report(y_test, y_pred))

Các chỉ số Accuracy, Precision, Recall và F1_score cho thấy mô hình ở nhiều góc độ khác nhau. Nhưng F1_score thường được quan tâm vì cân bằng giữa Precision và Recall.

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Không sống", "Sống"],
    yticklabels=["Không sống", "Sống"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix của bài 1")
plt.show()

## Nhận xét biểu đồ:
Có (96 + 47)/179 = 143/179 = 79.9% mẫu mô hình đã dự đoán đúng tương ứng với Accuracy, cho thấy khả năng phân loại của nó khá tốt. Nhưng độ nhận diện hành khách không sống sót (96) tốt hơn hành khách sống sót (47). Nhìn chung, mô hình phân loại khá chính xác trên tệp dữ liệu này.

In [ ]:
# So sánh với Linear Regression
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])
linear_model.fit(X_train, y_train)

# Dự đoán
y_pred_linear = linear_model.predict(X_test)

# Chuyển về 0 hoặc 1
y_pred_linear = (y_pred_linear >= 0.5).astype(int)
acc_linear = accuracy_score(y_test, y_pred_linear)
pre_linear = precision_score(y_test, y_pred_linear)
rec_linear = recall_score(y_test, y_pred_linear)
f1_linear = f1_score(y_test, y_pred_linear)

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1_score"],
    "Linear Regression": [
        acc_linear,
        pre_linear,
        rec_linear,
        f1_linear
    ],
    "Logistic Regression": [
        acc,
        pre,
        rec,
        f1
    ]
})
comparison

## So sánh 2 mô hình Logistic Regression và Linear Regression.
Logistic Regression thường có các chỉ số cao hơn hẳn so với Linear Regression. Logistic Regression dự đoán xác suất bằng hàm Sigmoid, phù hợp với bài toán phân loại nhị phân; còn Linear Regression dự đoán giá trị liên tục nên phải chuyển thành 0 hoặc 1 bằng 1 ngưỡng khác, dẫn đến hiệu quả thấp hơn. 
=> Đối với bộ dữ liệu Titanic này, Logistic Regression phù hợp hơn để dự đoán khả năng sống sót của hành khách trên tàu.

# Bài 2: Dry Bean Dataset
## Xây dựng mô hình phân loại các loại hạt trong bộ dữ liệu Dry Bean Dataset bằng hai thuật toán:

Logistic Regression
K-Nearest Neighbors – KNN
Dữ liệu
## Dữ liệu đã được preprocessing (file processing_seeds.ipynb) và chia thành hai tập:

Dry_Bean_Dataset/
├── dry_bean_train.csv
└── dry_bean_test.csv

## Các bạn có thể tự tạo file processing để có thể processing theo ý mình

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

train_df = pd.read_csv("Dry_bean_dataset/dry_bean_train.csv")
test_df = pd.read_csv("Dry_bean_dataset/dry_bean_test.csv")
print("Dữ liệu của train", train_df.shape)
print("Dữ liệu của test:", test_df.shape)
train_df.head()

In [ ]:
# X chứa các đặc trưng.
# y là loại hạt cần dự đoán.
X_train = train_df.drop("Class", axis=1)
y_train = train_df["Class"]
X_test = test_df.drop("Class", axis=1)
y_test = test_df["Class"]
print(X_train.shape)
print(X_test.shape)

## Huấn luyện Logistic Regression.

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
log_model.fit(X_train, y_train)

# Dự đoán.
y_pred_log = log_model.predict(X_test)

## Đánh giá Logistic Regression.

In [ ]:
acc = accuracy_score(y_test, y_pred_log)
pre = precision_score(y_test, y_pred_log, average="weighted")
rec = recall_score(y_test, y_pred_log, average="weighted")
f1 = f1_score(y_test, y_pred_log, average="weighted")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1_score:", f1)
print(classification_report(y_test, y_pred_log))
# bài toán đa lớp nên phải tính trung bình có trọng số giữa các lớp.

## Biểu đồ thể hiện Confusion Matrix của Logistic.

In [ ]:
cm = confusion_matrix(y_test, y_pred_log)
plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.title("Confusion Matrix của bài 2 - Logistic")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

Mô hình đạt độ chính xác khá cao, ma trận nhầm lẫn cho thấy phần lớn mẫu được dự đoán là đúng, chỉ có thiểu số bị nhầm giữa các lớp có đặc điểm tương tự.

## Huấn luyện KNN.

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5 # một hạt đậu mới sẽ được phân theo 5 hạt đậu gần nhất.
)
knn.fit(X_train, y_train)

# Dự đoán.
y_pred_knn = knn.predict(X_test)

## Đánh giá KNN.

In [ ]:
acc = accuracy_score(y_test, y_pred_knn)
pre = precision_score(y_test, y_pred_knn, average="weighted")
rec = recall_score(y_test, y_pred_knn, average="weighted")
f1 = f1_score(y_test, y_pred_knn, average="weighted")
print("Accuracy:", acc)
print("Precision:", pre)
print("Recall:", rec)
print("F1_score:", f1)
print(classification_report(y_test, y_pred_knn))

## Biểu đồ thể hiện Confusion Matrix của KNN.

In [ ]:
cm = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Reds"
)
plt.title("Confusion Matrix của bài 2 - KNN")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

Mô hình phân loại dựa trên các mẫu lân cận gần nhất. Nếu dữ liệu được chuẩn hóa trong tiền xử lý, KNN cho độ chính xác cao, do khoảng cách giữa các điểm dữ liệu phản ánh đúng mức độ tương đồng giữa các mẫu.

## So sánh 2 mô hình (thêm cho biết)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN"]
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_knn)
    ],
    "Precision": [
        precision_score(y_test, y_pred_log, average="weighted"),
        precision_score(y_test, y_pred_knn, average="weighted")
    ],
    "Recall": [
        recall_score(y_test, y_pred_log, average="weighted"),
        recall_score(y_test, y_pred_knn, average="weighted")
    ],
    "F1_score": [
        f1_score(y_test, y_pred_log, average="weighted"),
        f1_score(y_test, y_pred_knn, average="weighted")
    ]
})
comparison
# Vẽ biểu đồ so sánh.
comparison.set_index("Model").plot(
    kind="bar",
    figsize=(6,4)
)
plt.title("So sánh giữa Logistic Regression và KNN")
plt.ylabel("Score")
plt.grid(axis="y")
plt.show()

Logistic Regression có mô hình đơn giản và quy trình huấn luyện nhẹ nhàng, nhanh.
KNN có các chỉ số cao hơn, đặc biệt ở Accuracy và F1_score cao hơn hẳn nên rất phù hợp với dữ liệu do khai thác tốt cấu trúc gần nhất của các mẫu. 
=> Tùy bài toán sẽ phù hợp với mô hình nào.